# Pixels to Predictions — 0.82 Solution

**Key ideas that got 0.8229:**
- Train on **train + val combined** (4,157 samples instead of 3,109)
- LoRA r=8, alpha=32 (4x ratio), no DoRA
- Subject / grade / lecture / hint in prompt
- Letter-only NLL scoring at inference (fast, reliable)
- 3 epochs, LR=1.5e-4, img=512

**Runtime:** ~8-9h on single T4 · ~4-5h on A100  
**Kaggle:** Enable GPU accelerator → set to T4  
**Colab:** Runtime → Change runtime type → T4 GPU

In [ ]:
# Install dependencies
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes==0.45.5 accelerate

In [ ]:
import os, json, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset

warnings.filterwarnings('ignore')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Paths — adjust if your dataset is in a different location ──
if os.path.exists('/kaggle/input'):
    # Kaggle: adjust the folder name to match your dataset
    DATA_DIR   = Path('/kaggle/input/pixels-to-predictions')
    OUTPUT_DIR = Path('/kaggle/working')
else:
    # Colab: upload your data folder to /content/data
    DATA_DIR   = Path('/content/data')
    OUTPUT_DIR = Path('/content/output')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_DIR = DATA_DIR / 'images' / 'images'
print(f'Data dir   : {DATA_DIR}  (exists: {DATA_DIR.exists()})')
print(f'Image dir  : {IMAGE_DIR}  (exists: {IMAGE_DIR.exists()})')
print(f'Output dir : {OUTPUT_DIR}')

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────
SEED                 = 42
MODEL_ID             = 'HuggingFaceTB/SmolVLM-500M-Instruct'
IMG_MAX_SIZE         = 512
NUM_EPOCHS           = 3       # reduce to 2 if running out of time
PER_DEVICE_BATCH     = 1
GRAD_ACCUM_STEPS     = 8
LEARNING_RATE        = 1.5e-4
WARMUP_RATIO         = 0.05
WEIGHT_DECAY         = 0.01
LORA_R               = 8
LORA_ALPHA           = 32      # 4x ratio — key to the 0.82 score
LORA_DROPOUT         = 0.05
CHOICE_LETTERS       = 'ABCDEFGHIJ'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Config set.')

In [ ]:
# ── Load data ──────────────────────────────────────────────────
def parse_choices(x):
    if isinstance(x, list): return x
    if pd.isna(x): return []
    try: return json.loads(x)
    except Exception:
        import ast
        return ast.literal_eval(x)

train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df   = pd.read_csv(DATA_DIR / 'val.csv')
test_df  = pd.read_csv(DATA_DIR / 'test.csv')

for df in [train_df, val_df, test_df]:
    df['choices'] = df['choices'].apply(parse_choices)

# Combine train + val for training (key improvement!)
all_train_df = pd.concat([train_df, val_df], ignore_index=True)

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Combined: {len(all_train_df):,} | Test: {len(test_df):,}')

In [ ]:
# ── Image handling ─────────────────────────────────────────────
IMAGE_EXTS  = {'.png', '.jpg', '.jpeg', '.webp'}
IMAGE_INDEX = {}
for p in IMAGE_DIR.rglob('*'):
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
        IMAGE_INDEX[p.name] = p
        IMAGE_INDEX[str(p.relative_to(IMAGE_DIR))] = p
        try:
            IMAGE_INDEX[str(p.relative_to(DATA_DIR))] = p
        except ValueError:
            pass
print(f'Indexed {len(IMAGE_INDEX):,} image entries')

def resolve_image_path(row):
    raw = str(row['image_path'])
    raw_obj = Path(raw)
    for c in [DATA_DIR / raw, IMAGE_DIR / raw, IMAGE_DIR / raw_obj.name]:
        if c.exists(): return c
    for k in [raw, raw_obj.name]:
        if k in IMAGE_INDEX: return IMAGE_INDEX[k]
    raise FileNotFoundError(f'Cannot find: {raw}')

def load_image(row):
    img = Image.open(resolve_image_path(row)).convert('RGB')
    img.thumbnail((IMG_MAX_SIZE, IMG_MAX_SIZE), Image.Resampling.BICUBIC)
    canvas = Image.new('RGB', (IMG_MAX_SIZE, IMG_MAX_SIZE), 'white')
    canvas.paste(img, ((IMG_MAX_SIZE - img.width) // 2, (IMG_MAX_SIZE - img.height) // 2))
    return canvas

In [ ]:
# ── Prompt builder ─────────────────────────────────────────────
def clean_text(x):
    if x is None or pd.isna(x): return ''
    return ' '.join(str(x).replace('\n', ' ').split())

def build_prompt(row, include_answer=False):
    parts = ['<image>', 'Look at the image and answer the science multiple-choice question.']

    subject = clean_text(row.get('subject', ''))
    grade   = clean_text(row.get('grade', ''))
    if subject or grade:
        parts.append(f'Subject: {subject}  Grade: {grade}')

    lecture = clean_text(row.get('lecture', ''))
    hint    = clean_text(row.get('hint', ''))
    if lecture: parts.append(f'Background: {lecture}')
    if hint:    parts.append(f'Hint: {hint}')

    parts.append(f'Question: {clean_text(row["question"])}')
    choices_text = '\n'.join(
        [f'{CHOICE_LETTERS[i]}. {clean_text(c)}' for i, c in enumerate(row['choices'])]
    )
    parts.append(f'Choices:\n{choices_text}')
    parts.append('The correct answer is:')

    prompt = '\n\n'.join(parts)
    if include_answer:
        prompt += ' ' + CHOICE_LETTERS[int(row['answer'])]
    return prompt

# Quick sanity check
sample = all_train_df.iloc[0]
print(build_prompt(sample)[:400], '...')

In [ ]:
# ── Load model with QLoRA (4-bit) ──────────────────────────────
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {n_trainable:,}  (cap: 5,000,000)')
assert n_trainable <= 5_000_000, f'Exceeds 5M cap: {n_trainable:,}'

In [ ]:
# ── Dataset + collator ─────────────────────────────────────────
class ScienceQADataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {'row': row, 'image': load_image(row)}

def train_collate_fn(batch):
    images  = [b['image'] for b in batch]
    prompts = [build_prompt(b['row'], include_answer=False) for b in batch]
    answers = [' ' + CHOICE_LETTERS[int(b['row']['answer'])] for b in batch]
    eos     = processor.tokenizer.eos_token or ''
    full_texts = [p + a + eos for p, a in zip(prompts, answers)]

    full   = processor(text=full_texts, images=images, padding=True, return_tensors='pt')
    p_only = processor(text=prompts,    images=images, padding=True, return_tensors='pt')

    labels = full['input_ids'].clone()
    labels[full['attention_mask'] == 0] = -100
    for i in range(len(batch)):
        prompt_len = int(p_only['attention_mask'][i].sum().item())
        labels[i, :prompt_len] = -100

    full['labels'] = labels
    return full

train_dataset = ScienceQADataset(all_train_df)
print(f'Training samples: {len(train_dataset):,}')

In [ ]:
# ── Training ───────────────────────────────────────────────────
from transformers import Trainer, TrainingArguments, TrainerCallback

ADAPTER_DIR    = OUTPUT_DIR / 'smolvlm_adapter'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

class SaveCallback(TrainerCallback):
    """Mark each checkpoint so we can safely resume."""
    def on_save(self, args, state, control, **kwargs):
        ckpt = Path(args.output_dir) / f'checkpoint-{state.global_step}'
        (ckpt / 'plain_lora.txt').write_text('true')

# Resume from last checkpoint if it exists
existing = sorted(CHECKPOINT_DIR.glob('checkpoint-*'))
resume_from = None
if existing:
    last = existing[-1]
    if (last / 'plain_lora.txt').exists():
        resume_from = str(last)
        print(f'Resuming from: {resume_from}')
    else:
        import shutil
        for c in existing: shutil.rmtree(c, ignore_errors=True)
        print('Cleared old checkpoints')
if not resume_from:
    print('Starting training from scratch')

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type='cosine',
    optim='paged_adamw_8bit',
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=20,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=2,
    report_to='none',
    remove_unused_columns=False,
    dataloader_num_workers=2,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=train_collate_fn,
    callbacks=[SaveCallback()],
)

trainer.train(resume_from_checkpoint=resume_from)

model.save_pretrained(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)
print(f'Adapter saved to: {ADAPTER_DIR}')

In [ ]:
# ── Inference — letter-only NLL scoring ────────────────────────
model.eval()

def move_to_device(batch):
    dev = next(model.parameters()).device
    return {k: v.to(dev) if torch.is_tensor(v) else v for k, v in batch.items()}

@torch.inference_mode()
def predict_row(row):
    image  = load_image(row)
    prompt = build_prompt(row)
    n      = len(row['choices'])
    texts  = [prompt + ' ' + CHOICE_LETTERS[i] for i in range(n)]
    imgs   = [image] * n

    enc  = processor(text=texts, images=imgs, padding=True, return_tensors='pt')
    penc = processor(text=[prompt]*n, images=imgs, padding=True, return_tensors='pt')
    enc  = move_to_device(enc)

    out    = model(**enc)
    logits = out.logits.float().cpu()
    ids    = enc['input_ids'].cpu()
    mask   = enc['attention_mask'].cpu()
    pmask  = penc['attention_mask'].cpu()

    scores = []
    for i in range(n):
        tlen = int(mask[i].sum().item())
        plen = int(pmask[i].sum().item())
        if plen >= tlen:
            scores.append(1e9)
            continue
        tgt = ids[i, plen:tlen]
        lp  = F.log_softmax(logits[i, plen-1:tlen-1, :], dim=-1)
        scores.append(-lp[torch.arange(len(tgt)), tgt].mean().item())
    return int(np.argmin(scores))

# Resume test predictions if checkpoint exists
TEST_CKPT = OUTPUT_DIR / 'test_predictions.json'
completed = {}
if TEST_CKPT.exists():
    with open(TEST_CKPT) as f:
        completed = json.load(f)
    print(f'Resuming: {len(completed)}/{len(test_df)} done')
else:
    print('Starting fresh prediction')

for idx in tqdm(range(len(test_df)), desc='Predicting'):
    row    = test_df.iloc[idx]
    row_id = str(row['id'])
    if row_id in completed:
        continue
    completed[row_id] = predict_row(row)
    if len(completed) % 50 == 0:
        with open(TEST_CKPT, 'w') as f:
            json.dump(completed, f)
        print(f'  Saved {len(completed)}/{len(test_df)}')

with open(TEST_CKPT, 'w') as f:
    json.dump(completed, f)
print('Prediction complete!')

In [ ]:
# ── Build and validate submission ──────────────────────────────
submission = pd.DataFrame({
    'id':     test_df['id'].astype(str),
    'answer': [completed[str(r)] for r in test_df['id']],
})

assert list(submission.columns) == ['id', 'answer']
assert len(submission) == len(test_df)
assert submission['id'].is_unique
for pred, choices in zip(submission['answer'], test_df['choices']):
    assert 0 <= int(pred) < len(choices), f'Invalid pred {pred}'

sub_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(sub_path, index=False)
print(f'Saved: {sub_path}')
print(f'\nAnswer distribution:')
print(submission['answer'].value_counts().sort_index())
print(f'\nFirst 10 rows:')
print(submission.head(10))